# RAIW 10-provider source-pipeline classifier (`multi-provider-source-v8`)

This notebook demonstrates the unified (`57.80 MB`), CPU-only, pickle-free (`allow_pickle=False`) multimodal classifier published at [`wiltodelta/raiw-source-classify`](https://huggingface.co/wiltodelta/raiw-source-classify).

- **10 AI provider families + open-set abstention:** `openai`, `google`, `bytedance`, `flux`, `stable_diffusion`, `meta`, `midjourney`, `ideogram`, `firefly`, `xai`, plus `unknown`.
- **Stage-2 model/version attribution (`version_subfamily`):** identifies specific generator subfamilies when an image clears the calibrated regime-aware acceptance threshold.
- **Calibrated binary synthetic detection (`synthetic_detected`, `synthetic_probability`):** retains synthetic recall across foreign resize and container transcoding while abstaining from unidentifiable generator attribution (`label="unknown"`).
- **Scope boundary:** pixel source-pipeline classification is **not** provenance verification and **not** a SynthID or invisible-watermark detector (`watermark_truth="unknown"`).

In [ ]:
%pip install -q "remove-ai-watermarks[source-classify]>=0.64.0"

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
from huggingface_hub import hf_hub_download

from remove_ai_watermarks import source_classify as sc

weights_path = Path(
    hf_hub_download(
        repo_id=sc.WEIGHTS_REPO,
        filename=sc.MODEL_FILE,
        revision=sc.WEIGHTS_REVISION,
    )
)
with weights_path.open("rb") as handle:
    sha256 = hashlib.file_digest(handle, "sha256").hexdigest()
if sha256 != sc.MODEL_SHA256:
    raise ValueError(f"Unexpected SHA-256: {sha256}")

with np.load(weights_path, allow_pickle=False) as bundle:
    schema = str(np.asarray(bundle["schema"]).item())
    providers = [str(x) for x in np.asarray(bundle["provider_classes"]).tolist()]
    stage2_heads = {
        str(p): [str(s) for s in np.asarray(bundle[f"s2_{p}_subfamilies"]).tolist()]
        for p in np.asarray(bundle["stage2_version_classes"]).tolist()
    }

sys.stdout.write(
    json.dumps(
        {
            "artifact": weights_path.name,
            "size_mb": round(weights_path.stat().st_size / (1024 * 1024), 2),
            "sha256": sha256,
            "schema": schema,
            "providers": providers,
            "stage2_version_heads": stage2_heads,
        },
        indent=2,
    )
    + "\n"
)

In [ ]:
from PIL import Image, ImageDraw

import remove_ai_watermarks as raiw

# Construct a clean non-AI procedural graphic (chart / UI card)
canvas = Image.new("RGB", (768, 512), color=(248, 250, 252))
draw = ImageDraw.Draw(canvas)
draw.rectangle((48, 48, 720, 464), outline=(148, 163, 184), width=3)
for idx, bar_h in enumerate((120, 210, 165, 290, 245, 330)):
    x0 = 96 + idx * 96
    draw.rectangle((x0, 420 - bar_h, x0 + 56, 420), fill=(59, 130, 246))

res = raiw.classify_source(canvas)
sys.stdout.write(
    json.dumps(
        {
            "label": res.label,
            "version_subfamily": res.version_subfamily,
            "synthetic_detected": res.synthetic_detected,
            "synthetic_probability": round(res.synthetic_probability, 4),
            "reason": res.reason,
            "confidence": round(res.confidence, 4),
            "watermark_truth": res.watermark_truth,
        },
        indent=2,
    )
    + "\n"
)

In [ ]:
from pathlib import Path

# Set IMAGE_PATH to any local PNG/JPEG/WebP file (or upload one in Colab's Files sidebar)
IMAGE_PATH = ""  # e.g. "sample.jpg"

if IMAGE_PATH and Path(IMAGE_PATH).is_file():
    prediction = raiw.classify_source(IMAGE_PATH)
    sys.stdout.write(json.dumps(prediction.to_dict(), indent=2) + "\n")
else:
    sys.stdout.write("Set IMAGE_PATH to an image file to inspect its provider and version_subfamily prediction.\n")